# BÀI TẬP: MEDICAL INSURANCE COST
**Nguồn:** kaggle.com/datasets/mosapabdelghany/medical-insurance-cost-dataset



## Setup

In [36]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from scipy import stats

sns.set_style('whitegrid')


csv_path = 'https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv'

df = pd.read_csv(csv_path)
print('Loaded from:', csv_path)
df.head()

Loaded from: https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv


,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


---
# PHẦN A — DATA PROFILING
## A.1. Data size, column names, data types

In [37]:
# TODO
print("data size:", df.shape)
print("column names:", df.columns.tolist())
print("data types:")
print(df.dtypes)

data size: (1338, 7)
column names: ['age', 'sex', 'bmi', 'children', 'smoker', 'region', 'charges']
data types:
age           int64
sex             str
bmi         float64
children      int64
smoker          str
region          str
charges     float64
dtype: object


## A.2. Missing values & Duplicate data

In [38]:
# TODO
print("missing values:")
print(df.isnull().sum())
print("duplicates:", df.duplicated().sum())

missing values:
age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64
duplicates: 1


## A.3. Invalid values

In [39]:
# TODO
print("age <= 0:", (df['age'] <= 0).sum())
print("bmi <= 0:", (df['bmi'] <= 0).sum())
print("charges < 0:", (df['charges'] < 0).sum())

age <= 0: 0
bmi <= 0: 0
charges < 0: 0


## A.4. Create a new column
Tạo cột `bmi_group`: Normal (<25), Overweight (25-30), Obese (>=30).

In [40]:
# TODO
def classify_bmi(bmi):
    if bmi < 25:
        return 'Normal'
    elif bmi < 30:
        return 'Overweight'
    else:
        return 'Obese'

df['bmi_group'] = df['bmi'].apply(classify_bmi)
df.head()

,age,sex,bmi,children,smoker,region,charges,bmi_group
0,19,female,27.900,0,yes,southwest,16884.92400,Overweight
1,18,male,33.770,1,no,southeast,1725.55230,Obese
2,28,male,33.000,3,no,southeast,4449.46200,Obese
3,33,male,22.705,0,no,northwest,21984.47061,Normal
4,32,male,28.880,0,no,northwest,3866.85520,Overweight


---
# PHẦN B — DESCRIPTIVE STATISTICS
## Group 1 — Central Tendency

In [41]:
# TODO
numeric_cols = ['age', 'bmi', 'children', 'charges']
print("mean")
print(df[numeric_cols].mean())
print("median")
print(df[numeric_cols].median())

mean
age            39.207025
bmi            30.663397
children        1.094918
charges     13270.422265
dtype: float64
median
age           39.000
bmi           30.400
children       1.000
charges     9382.033
dtype: float64


## Group 2 — Dispersion

In [42]:
# TODO
print("Độ lệch chuẩn")
print(df[numeric_cols].std())
print(" IQR (Interquartile Range)")
Q1 = df[numeric_cols].quantile(0.25)
Q3 = df[numeric_cols].quantile(0.75)
print(Q3 - Q1)

Độ lệch chuẩn
age            14.049960
bmi             6.098187
children        1.205493
charges     12110.011237
dtype: float64
 IQR (Interquartile Range)
age            24.000000
bmi             8.397500
children        2.000000
charges     11899.625365
dtype: float64


## Group 3 — Location and Shape

In [43]:
# TODO
print("(Độ lệch)")
print(df[numeric_cols].skew())

(Độ lệch)
age         0.055673
bmi         0.284047
children    0.938380
charges     1.515880
dtype: float64


---
# PHẦN C — DEFINE THE QUESTION

## Câu hỏi 1: Người hút thuốc trả chi phí cao hơn bao nhiêu lần so với người không hút, và có đồng đều giữa các vùng không?

In [44]:
# TODO
mean_charges = df.groupby('smoker')['charges'].mean()
print("Chi phí trung bình theo smoker:\n", mean_charges)
print(f"Người hút thuốc trả cao gấp {mean_charges['yes'] / mean_charges['no']:.2f} lần người không hút.")

region_smoker = df.pivot_table(index='region', columns='smoker', values='charges', aggfunc='mean')
region_smoker['ratio'] = region_smoker['yes'] / region_smoker['no']
print("\nChi phí theo vùng và trạng thái hút thuốc:\n", region_smoker)

Chi phí trung bình theo smoker:
 smoker
no      8434.268298
yes    32050.231832
Name: charges, dtype: float64
Người hút thuốc trả cao gấp 3.80 lần người không hút.

Chi phí theo vùng và trạng thái hút thuốc:
 smoker              no           yes     ratio
region                                        
northeast  9165.531672  29673.536473  3.237514
northwest  8556.463715  30192.003182  3.528561
southeast  8032.216309  34844.996824  4.338155
southwest  8019.284513  32269.063494  4.023933


## Câu hỏi 2: BMI có tương quan với chi phí mạnh hơn ở nhóm hút thuốc hay không hút thuốc?

In [45]:
# TODO
smoker_corr = df[df['smoker'] == 'yes'][['bmi', 'charges']].corr().iloc[0, 1]
nonsmoker_corr = df[df['smoker'] == 'no'][['bmi', 'charges']].corr().iloc[0, 1]
print(f"Hệ số tương quan BMI & Charges (Hút thuốc): {smoker_corr:.4f}")
print(f"Hệ số tương quan BMI & Charges (Không hút thuốc): {nonsmoker_corr:.4f}")

Hệ số tương quan BMI & Charges (Hút thuốc): 0.8065
Hệ số tương quan BMI & Charges (Không hút thuốc): 0.0840


## Câu hỏi 3: Vùng nào có chi phí bảo hiểm trung bình cao nhất?

In [46]:
# TODO
avg_region = df.groupby('region')['charges'].mean().sort_values(ascending=False)
print("Chi phí bảo hiểm trung bình theo vùng:\n", avg_region)
print(f"Vùng có chi phí cao nhất là: {avg_region.index[0]}")

Chi phí bảo hiểm trung bình theo vùng:
 region
southeast    14735.411438
northeast    13406.384516
northwest    12417.575374
southwest    12346.937377
Name: charges, dtype: float64
Vùng có chi phí cao nhất là: southeast


## Câu hỏi 4: Số lượng con cái có làm tăng chi phí bảo hiểm không?

In [47]:
# TODO
print("Chi phí bảo hiểm trung bình theo số con:\n", df.groupby('children')['charges'].mean())

Chi phí bảo hiểm trung bình theo số con:
 children
0    12365.975602
1    12731.171832
2    15073.563734
3    15355.318367
4    13850.656311
5     8786.035247
Name: charges, dtype: float64


## Câu hỏi 5: Tuổi có tương quan với chi phí bảo hiểm không?

In [48]:
# TODO
age_corr = df[['age', 'charges']].corr().iloc[0, 1]
print(f"Hệ số tương quan giữa Tuổi và Chi phí: {age_corr:.4f}")

Hệ số tương quan giữa Tuổi và Chi phí: 0.2990


## Câu hỏi 6 (Tổng hợp) — Viết insight tổng hợp
Dựa trên Phần A, B, C, viết 4-5 câu insight tổng thể.

*(Viết insight của bạn vào đây...)*